# Volet B — Comparaison des encodeurs pour la recherche de clauses

**Ce que fait ce notebook** (il n'appelle que `src/ledgar/encoders.py`) : 7 encodeurs de phrases pré-entraînés, sans entraînement, évalués comme dans `ledgar.retrieval` : banque = train, requêtes = validation (choix) et test (chiffres finaux), une clause est pertinente si elle a le type de la requête, jumeaux exacts retirés.

Chaque encodeur est mesuré **seul** et **fusionné avec TF-IDF** (alpha × encodeur + (1 − alpha) × TF-IDF, alpha choisi sur la validation).

**Avant de lancer** : Exécution > Modifier le type d'exécution > **GPU T4**, puis déposer `ledgar_repo.zip` dans l'onglet Fichiers (icône dossier à gauche), puis **Exécution > Tout exécuter**. Durée : environ 45 min.

In [ ]:
%cd /content
import os, sys
# Code du repo : ledgar_repo.zip déposé dans l'onglet Fichiers, sinon clone avec un token GitHub (secret Colab GH_TOKEN)
if not os.path.exists("ledgar-contract-review"):
    if os.path.exists("ledgar_repo.zip"):
        !unzip -q ledgar_repo.zip -d ledgar-contract-review
    else:
        from google.colab import userdata
        TOKEN = userdata.get("GH_TOKEN")
        !git clone -b feat/retrieval https://{TOKEN}@github.com/saouss-07/ledgar-contract-review.git
%cd /content/ledgar-contract-review
!pip install -q -r requirements.txt && pip install -q -e .
sys.path.insert(0, "/content/ledgar-contract-review/src")   # pip install -e needs a restart otherwise
import torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4"
print("GPU :", torch.cuda.get_device_name(0))

## 1 — Encodage et évaluation des 7 encodeurs

Les embeddings de chaque modèle sont mis en cache : si Colab se déconnecte, relancer cette cellule reprend où elle s'était arrêtée (tant que la session n'est pas supprimée).

In [ ]:
!python -m ledgar.encoders

## 2 — Résultats sur le test

In [ ]:
import pandas as pd
t = pd.read_csv("results/tables/retrieval_encoders.csv")
cols = ["model", "method", "alpha", "P@1", "P@5", "P@10", "MRR@10", "params_M", "dim", "encode_train_s"]
t[t.split == "test"][cols].round(3)

In [ ]:
from IPython.display import Image
Image("results/figures/retrieval_encoders.png", width=750)

## 3 — Choix d'alpha sur la validation

In [ ]:
t[t.split == "validation"][["model", "method", "alpha", "P@5", "MRR@10"]].round(3)

## 4 — Télécharger les résultats

Envoyer `results_encoders.zip` pour qu'il soit ajouté au repo.

In [ ]:
!zip -q results_encoders.zip results/tables/retrieval_encoders.csv results/figures/retrieval_encoders.png
from google.colab import files
files.download("results_encoders.zip")